# Speculative decoding on a free T4 GPU

Runs every benchmark (correctness, hardware profile, draft-size sweep, calibration, language study) on the Colab T4 and saves the results so they can be merged with the laptop results.

**Before running:** `Runtime → Change runtime type → T4 GPU`. Then `Runtime → Run all`. Takes roughly 30–45 minutes.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 1. Get the code
Clones the GitHub repo. Set `REPO_URL = ""` to upload a zip of the project instead.

In [ ]:
import os, glob, zipfile
REPO_URL = "https://github.com/AdityaRaj272/unispec-everyday-hardware-indic.git"   # or "" to upload a zip instead

os.chdir("/content")
if REPO_URL:
    !git clone -q {REPO_URL} project
else:
    from google.colab import files
    up = files.upload()                      # upload the zipped Implementation folder
    with zipfile.ZipFile(next(iter(up))) as z:
        z.extractall("project")

# move into the folder that contains run.py
root = os.path.dirname(sorted(glob.glob("/content/project/**/run.py", recursive=True), key=len)[0])
os.chdir(root)
print("working in", root)

In [ ]:
!pip install -q -r requirements.txt
!python tests/test_engine.py

## 2. Run all benchmarks
Device, precision (fp16 on T4) and workload are picked automatically. Finished steps are skipped if the cell is re-run, so a disconnect only loses the step in progress.

In [ ]:
!python -u run.py all

### Optional: a bigger model (1.5B) on the same GPU
Shows how model size changes the hardware fingerprint. Results go to a separate `__qwen1.5b` folder.

In [ ]:
# !python -u run.py correctness profile sweep calibrate --model Qwen/Qwen2.5-1.5B-Instruct --tag qwen1.5b

## 3. Figures (includes any other devices already in `results/`)

In [ ]:
!python run.py plots
from IPython.display import Image, Markdown, display
for p in sorted(glob.glob("figures/*.png")):
    display(Image(p))
display(Markdown(open("results/SUMMARY.md", encoding="utf-8").read()))

## 4. Download the results
Unzip into the local `Implementation/` folder (it contains `results/cuda-tesla-t4/`), then run `python run.py plots` locally.

In [ ]:
!zip -qr t4_results.zip results figures
from google.colab import files
files.download("t4_results.zip")